# Multi-Source Production Environment Test

This notebook exercises every production retrieval index (domain knowledge, Mail, and Calendar), a combined multi-source query, and multi-turn memory through the real `/v1/chat` application path. Set `USER_ID` to a trusted test owner whose production data may be queried.

In [ ]:
import os
import sys
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "app" / "api" / "main.py").is_file():
            return candidate
    raise RuntimeError("Repository root containing app/api/main.py was not found")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
os.chdir(REPOSITORY_ROOT)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
print(f"Repository: {REPOSITORY_ROOT}")

## Load production configuration safely

Secrets are read from the process environment or the repository-root `.env`. Secret values are never printed; only safe endpoint metadata and OpenRouter key presence are displayed.

In [ ]:
from pprint import pprint
from urllib.parse import urlsplit

from app.config.settings import Settings


def safe_endpoint_description(endpoint: str) -> str:
    parsed = urlsplit(endpoint)
    scheme = parsed.scheme if parsed.scheme in {"http", "https"} else "configured"
    hostname = parsed.hostname or "configured-host"
    if ":" in hostname:
        hostname = f"[{hostname}]"
    try:
        port = parsed.port
    except ValueError:
        port = None
    if port is not None:
        hostname = f"{hostname}:{port}"
    return f"{scheme}://{hostname}"


settings = Settings.from_env()
llm_endpoint = settings.resolve_llm_endpoint()
embedding_endpoint = settings.resolve_embedding_endpoint()
safe_settings = {
    "llm_provider": llm_endpoint.provider,
    "llm_model": llm_endpoint.model,
    "llm_base_url": safe_endpoint_description(llm_endpoint.base_url),
    "embedding_provider": embedding_endpoint.provider,
    "embedding_model": embedding_endpoint.model,
    "embedding_base_url": safe_endpoint_description(embedding_endpoint.base_url),
    "mongo_db": settings.mongo_db,
    "domain_index": settings.domain_knowledge_index,
    "mail_alias": settings.mail_index_alias,
    "calendar_alias": settings.calendar_index_alias,
    "timezone": settings.default_user_timezone,
    "MULTI_SOURCE_DEMO": settings.multi_source_demo,
    "OPENROUTER_API_KEY configured": bool(
        settings.openrouter_api_key.get_secret_value().strip()
    ),
}
pprint(safe_settings)
assert settings.multi_source_demo is False, "Set MULTI_SOURCE_DEMO=false"
assert llm_endpoint.provider == "openrouter"
assert llm_endpoint.model == "openrouter/free"
assert safe_settings["OPENROUTER_API_KEY configured"], (
    "Set OPENROUTER_API_KEY in the process environment or repository .env"
)

## Build and check the production application

Application construction and readiness use the same production factories and dependency checks as the API service. The in-process ASGI client avoids an external HTTP server while preserving the real `/ready` and `/v1/chat` paths.

In [ ]:
import httpx

from app.api.dependencies import build_container
from app.api.main import create_app


container = build_container(settings)
application = create_app(container)
transport = httpx.ASGITransport(app=application, raise_app_exceptions=False)
client = httpx.AsyncClient(transport=transport, base_url="http://production-notebook")

readiness_response = await client.get("/ready")
readiness_body = readiness_response.json()
print("readiness HTTP", readiness_response.status_code)
pprint(readiness_body)
assert readiness_response.status_code == 200, (
    "Production dependencies are not ready; fix the reported dependency before chat"
)
assert readiness_body.get("status") == "ready"

## Configure all multi-index test scenarios

Each scenario calls the same public `/v1/chat` endpoint. Expected tools and source types are explicit test contracts, not keyword-based routing logic. Edit `USER_ID` only if the indexed test owner differs.

In [ ]:
USER_ID = "kim"
COMMON_FILTERS = {
    "teams": [],
    "weeks": [],
}
TEST_CASES = [
    {
        "name": "domain_knowledge",
        "message": "Cell Leakage가 뭐야?",
        "expected_tools": {"search_domain_knowledge"},
        "expected_source_types": {"domain_knowledge"},
    },
    {
        "name": "mail",
        "message": "NAND 관련 메일 찾아줘",
        "expected_tools": {"search_mail"},
        "expected_source_types": {"mail"},
    },
    {
        "name": "calendar",
        "message": "이번 주 일정 뭐야?",
        "expected_tools": {"search_calendar"},
        "expected_source_types": {"calendar"},
    },
    {
        "name": "multi_source",
        "message": (
            "김OO이 지난주 메일에서 이야기한 NAND 수율 문제가 어떤 회의에서 "
            "논의됐고 어떤 Action을 하기로 했으며 기술적으로 어떤 의미인지 설명해줘."
        ),
        "expected_tools": {"search_mail", "search_calendar", "search_domain_knowledge"},
        "expected_source_types": {"mail", "calendar", "domain_knowledge"},
    },
]
FOLLOW_UP = {
    "base_case": "multi_source",
    "message": "그 회의에서 결정한 Action과 관련 기술 의미를 다시 정리해줘.",
}

In [ ]:
obsolete_fields = {"response_mode", "mode", "routing"}


def show_chat_result(response: httpx.Response) -> dict:
    body = response.json()
    print("HTTP", response.status_code)
    print("x-trace-id", response.headers.get("x-trace-id", "missing"))
    if response.status_code != 200:
        pprint(body)
        raise AssertionError("Chat request failed; inspect the safe error and trace ID")

    assert obsolete_fields.isdisjoint(body)
    print("answer:", body.get("answer"))
    print("references:")
    for reference in body.get("references", []):
        print(
            f"- [{reference['evidence_id']}] {reference['source_type']} | "
            f"{reference['title']} | {reference['excerpt']}"
        )
    print("disclosures:", body.get("disclosures", []))
    print("agent_trace:", body.get("agent_trace"))
    execution = body.get("execution") or {}
    print(
        "execution:",
        {
            "status": execution.get("status"),
            "search_count": execution.get("search_count"),
            "evidence_count": execution.get("evidence_count"),
            "duration_ms": execution.get("duration_ms"),
        },
    )
    return body


def validate_chat_contract(body: dict, case: dict) -> None:
    trace = body.get("agent_trace") or {}
    llm_calls = trace.get("llm_calls") or []
    assert llm_calls[:2] == ["routing", "planner"]
    assert "judge" in llm_calls
    assert llm_calls[-1] == "answer"
    actual_tools = set(trace.get("tool_calls") or [])
    assert case["expected_tools"] <= actual_tools, (
        f"{case['name']}: expected tools {case['expected_tools']}, got {actual_tools}"
    )
    actual_source_types = {
        reference["source_type"] for reference in body.get("references", [])
    }
    assert case["expected_source_types"] <= actual_source_types, (
        f"{case['name']}: expected sources {case['expected_source_types']}, "
        f"got {actual_source_types}"
    )
    assert body["quality"]["citation_valid"] is True


chat_results = {}
conversation_ids = {}
for case in TEST_CASES:
    print(f"\n=== {case['name']} ===")
    print("question:", case["message"])
    chat_request = {
        "user_id": USER_ID,
        "message": case["message"],
        "filters": COMMON_FILTERS,
    }
    assert obsolete_fields.isdisjoint(chat_request)
    chat_response = await client.post("/v1/chat", json=chat_request)
    chat_body = show_chat_result(chat_response)
    validate_chat_contract(chat_body, case)
    chat_results[case["name"]] = chat_body
    conversation_ids[case["name"]] = chat_body["conversation_id"]

print("\nAll single-index and multi-source scenarios passed")

## Validate conversation continuity

The follow-up continues the combined multi-source conversation by reusing its server-issued conversation ID and owner, validating MongoDB-backed context continuity.

In [ ]:
conversation_id = conversation_ids[FOLLOW_UP["base_case"]]
follow_up_request = {
    "user_id": USER_ID,
    "message": FOLLOW_UP["message"],
    "conversation_id": conversation_id,
    "filters": COMMON_FILTERS,
}
assert obsolete_fields.isdisjoint(follow_up_request)
follow_up_response = await client.post("/v1/chat", json=follow_up_request)
follow_up_body = show_chat_result(follow_up_response)
assert follow_up_body["conversation_id"] == conversation_id

In [ ]:
await client.aclose()
print("Notebook HTTP client closed")